# AI Messages

The `ai.py` module defines AI-generated messages, streaming AI message chunks, standardized token-usage metadata, tool-call parsing, and utilities for combining message chunks and token counts.

# InputTokenDetails

`InputTokenDetails` is a `TypedDict` that stores a breakdown of input-token usage. Its fields are optional, do not need to sum to the total input-token count, and may be extended with provider-specific keys.

## Bases

- `TypedDict`

### Fields

1. `audio`: Stores the number of audio input tokens.
   * **Type:**
     ```python
     audio: int
     ```

2. `cache_creation`: Stores the number of input tokens used to create a cache after a cache miss.
   * **Type:**
     ```python
     cache_creation: int
     ```

3. `cache_read`: Stores the number of input tokens read from an existing cache after a cache hit.
   * **Type:**
     ```python
     cache_read: int
     ```

# OutputTokenDetails

`OutputTokenDetails` is a `TypedDict` that stores a breakdown of output-token usage. Its fields are optional, do not need to sum to the total output-token count, and may be extended with provider-specific keys.

## Bases

- `TypedDict`

### Fields

1. `audio`: Stores the number of audio output tokens.
   * **Type:**
     ```python
     audio: int
     ```

2. `reasoning`: Stores the number of reasoning tokens generated internally by the model but not returned as normal model output.
   * **Type:**
     ```python
     reasoning: int
     ```

# UsageMetadata

`UsageMetadata` is a standardized `TypedDict` for storing the input, output, and total token usage of an AI message.

## Bases

- `TypedDict`

### Fields

1. `input_tokens`: Stores the total number of input or prompt tokens.
   * **Type:**
     ```python
     input_tokens: int
     ```

2. `output_tokens`: Stores the total number of output or completion tokens.
   * **Type:**
     ```python
     output_tokens: int
     ```

3. `total_tokens`: Stores the combined number of input and output tokens.
   * **Type:**
     ```python
     total_tokens: int
     ```

4. `input_token_details`: Optionally stores a detailed breakdown of input-token usage.
   * **Type:**
     ```python
     input_token_details: NotRequired[InputTokenDetails]
     ```

5. `output_token_details`: Optionally stores a detailed breakdown of output-token usage.
   * **Type:**
     ```python
     output_token_details: NotRequired[OutputTokenDetails]
     ```

# AIMessage

`AIMessage` represents the response returned by a chat model. It stores the model's content together with standardized LangChain fields such as parsed tool calls, invalid tool calls, and usage metadata.

## Bases

- `BaseMessage`

### Attributes

1. `tool_calls`: Stores successfully parsed tool calls associated with the message.
   * **Type:**
     ```python
     tool_calls: list[ToolCall] = Field(default_factory=list)
     ```

2. `invalid_tool_calls`: Stores tool calls that could not be parsed successfully.
   * **Type:**
     ```python
     invalid_tool_calls: list[InvalidToolCall] = Field(default_factory=list)
     ```

3. `usage_metadata`: Stores standardized token-usage information for the message.
   * **Type:**
     ```python
     usage_metadata: UsageMetadata | None = None
     ```

4. `type`: Stores the message type used during serialization and deserialization.
   * **Type:**
     ```python
     type: Literal["ai"] = "ai"
     ```

### Properties

1. `lc_attributes`: Returns derived attributes that must be included during serialization.
   * **Type:**
     ```python
     lc_attributes: dict[str, Any]
     ```

2. `content_blocks`: Returns the message content as standardized, typed content blocks.

   It first uses a provider-specific content translator when available, then falls back to best-effort parsing. Missing tool-call blocks and reasoning blocks may be added to the result.

   * **Type:**
     ```python
     content_blocks: list[types.ContentBlock]
     ```

### Validators

1. `_backwards_compat_tool_calls`: Converts legacy tool-call data from `additional_kwargs` into standardized tool calls, invalid tool calls, or tool-call chunks.

   It also ensures that tool-call-like dictionaries are recreated with the correct standardized type.

   * **Syntax:**
     ```python
     @model_validator(mode="before")
     @classmethod
     _backwards_compat_tool_calls(
         cls,
         values: dict[str, Any] # Message values to validate and normalize
     ) -> Any
     ```

### Methods

1. `__init__`: Initializes an AI message from raw content or typed content blocks.

   When typed content blocks contain tool calls and no explicit `tool_calls` argument is supplied, those tool calls are automatically added to the message.

   * **Syntax:**
     ```python
     __init__(
         self,
         content: str | list[str | dict[Any, Any]] | None = None, # Raw message content
         content_blocks: list[types.ContentBlock] | None = None, # Typed standard content blocks
         **kwargs: Any # Additional arguments passed to BaseMessage
     ) -> None
     ```

2. `pretty_repr`: Returns a readable representation of the AI message.

   Parsed and invalid tool calls are appended with their names, IDs, arguments, and parsing errors.

   * **Syntax:**
     ```python
     pretty_repr(
         self,
         html: bool = False # Whether to return an HTML-formatted representation
     ) -> str
     ```

# AIMessageChunk

`AIMessageChunk` represents a partial AI message produced during streaming. Multiple chunks can be combined to form a complete AI message while preserving content, metadata, tool calls, token usage, and identifiers.

## Bases

- `AIMessage`
- `BaseMessageChunk`

### Attributes

1. `type`: Stores the chunk-specific message type used during serialization and deserialization.
   * **Type:**
     ```python
     type: Literal["AIMessageChunk"] = "AIMessageChunk"
     ```

2. `tool_call_chunks`: Stores partial tool-call information received during streaming.
   * **Type:**
     ```python
     tool_call_chunks: list[ToolCallChunk] = Field(default_factory=list)
     ```

3. `chunk_position`: Indicates whether the chunk represents the final aggregated position in a stream.
   * **Type:**
     ```python
     chunk_position: Literal["last"] | None = None
     ```

### Properties

1. `lc_attributes`: Returns derived tool-call attributes that must be included during serialization.
   * **Type:**
     ```python
     lc_attributes: dict[str, Any]
     ```

2. `content_blocks`: Returns the chunk content as standardized, typed content blocks.

   It can use a provider-specific chunk translator, convert tool-call chunks into standard content blocks, and insert extracted reasoning content when needed.

   * **Type:**
     ```python
     content_blocks: list[types.ContentBlock]
     ```

### Validators

1. `init_tool_calls`: Builds complete and invalid tool calls from the stored tool-call chunks.

   When the chunk is marked as the final chunk and uses output version `v1`, parsed tool-call chunks in the content are replaced with completed tool-call blocks.

   * **Syntax:**
     ```python
     @model_validator(mode="after")
     init_tool_calls(
         self
     ) -> Self
     ```

2. `init_server_tool_calls`: Converts final server-tool-call chunks with JSON string arguments into completed server-tool-call blocks.
   * **Syntax:**
     ```python
     @model_validator(mode="after")
     init_server_tool_calls(
         self
     ) -> Self
     ```

### Methods

1. `__add__`: Combines the current AI message chunk with another chunk or a sequence of chunks.

   AI message chunks are merged using `add_ai_message_chunks`. Other supported message values are delegated to `BaseMessageChunk`.

   * **Syntax:**
     ```python
     __add__(
         self,
         other: Any # AIMessageChunk, sequence of chunks, or another supported value
     ) -> BaseMessageChunk
     ```

## Functions

1. `add_ai_message_chunks`: Combines multiple AI message chunks into one chunk.

   It merges content, additional keyword arguments, response metadata, tool-call chunks, usage metadata, chunk position, and message IDs.

   * **Syntax:**
     ```python
     add_ai_message_chunks(
         left: AIMessageChunk, # First AI message chunk
         *others: AIMessageChunk # Additional AI message chunks
     ) -> AIMessageChunk
     ```

2. `add_usage`: Recursively adds two usage-metadata objects.

   When both values are missing, it returns zero token counts. When only one value is present, that value is returned.

   * **Syntax:**
     ```python
     add_usage(
         left: UsageMetadata | None, # First usage-metadata object
         right: UsageMetadata | None # Second usage-metadata object
     ) -> UsageMetadata
     ```

3. `subtract_usage`: Recursively subtracts one usage-metadata object from another.

   Token counts are prevented from becoming negative by applying `max(left - right, 0)` at each numeric field.

   * **Syntax:**
     ```python
     subtract_usage(
         left: UsageMetadata | None, # Usage metadata to subtract from
         right: UsageMetadata | None # Usage metadata to subtract
     ) -> UsageMetadata
     ```